In [0]:
from pyspark.sql import functions as F

BUCKET_NAME = "ride-sharing-dataplatform-aritra-2026"
SOURCE_PATH = f"s3://{BUCKET_NAME}/eventstream/trips/"
SCHEMA_PATH = f"s3://{BUCKET_NAME}/schemas/bronze_trips/"
CHECKPOINT_PATH = f"s3://{BUCKET_NAME}/checkpoints/bronze_trips/"
TARGET_TABLE = "default.bronze.trips"

In [0]:
print("========================================")
print("Starting Bronze Trip ingestion")
print("========================================")

print(f"Source      : {SOURCE_PATH}")
print(f"Schema path : {SCHEMA_PATH}")
print(f"Checkpoint  : {CHECKPOINT_PATH}")
print(f"Target      : {TARGET_TABLE}")

trips_stream = (
    spark.readStream
    # Auto Loader
    .format("cloudFiles")
    # Our simulator produces JSONL
    .option("cloudFiles.format", "json")
    # Persist Auto Loader's schema information
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)
    # Preserve unexpected columns/type mismatches
    .option("cloudFiles.schemaEvolutionMode","addNewColumnsWithTypeWidening")
    # Explicitly name the rescued data column
    .option("rescuedDataColumn", "_rescued_data")
    # Don't fail the stream just because one JSON
    # record contains a data-type mismatchs.
    .option("mode", "PERMISSIVE")
    .load(SOURCE_PATH)
)

# ============================================================
# Add ingestion metadata
# ============================================================

bronze_df = (
    trips_stream
    # When Spark processed the record
    .withColumn("_ingested_at", F.current_timestamp())
    # Source file that produced the record
    .withColumn("_source_file", F.col("_metadata.file_path"))
    # File modification time
    .withColumn("_source_file_modification_time", F.col("_metadata.file_modification_time"))
)


# ============================================================
# Write to Bronze Delta
# ============================================================

print("Starting Bronze Delta stream...")

query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    # Durable streaming checkpoint
    .option("checkpointLocation", CHECKPOINT_PATH)
    # Target Delta table
    .toTable(TARGET_TABLE)
)

print("Bronze ingestion stream started successfully.")